# 📊 Notebook 5 — Dashboard & Visualisation Report
**Input:** `speeches_final.csv` + `outputs/*.json`

All dashboard charts reproduced inline. Sections:
1. KPI summary
2. Time-of-day patterns
3. Per-politician & per-party analysis
4. RobBERT sentiment deep dive
5. Tone changes over the course of the day (per speaker)
6. Model comparison
7. Instructions to launch Streamlit dashboard


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib.ticker as mtick
import seaborn as sns, json, os, warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi':130,'axes.spines.top':False,'axes.spines.right':False,'figure.facecolor':'white'})
BLUE,RED,GREEN,ORANGE,GREY = '#2B5797','#C0392B','#27AE60','#E67E22','#95A5A6'
TONE_COLORS = {'aggressive':'#E74C3C','mean':'#E67E22','neutral':'#95A5A6','peaceful':'#3498DB','kind':'#27AE60','happy':'#F1C40F'}

df = pd.read_csv("speeches_final.csv", low_memory=False)
df['vergadering_datum'] = pd.to_datetime(df['vergadering_datum'], errors='coerce')
HAS_BERT = 'sentiment_label' in df.columns
print(f"Loaded {len(df):,} speeches | BERT columns: {HAS_BERT}")

## 1. KPI Summary

In [ ]:
total   = len(df); n_pass=int(df['label'].sum()); n_rej=total-n_pass
n_spk   = df['speaker_name'].nunique(); n_party=df['speaker_party'].nunique()
dr = f"{df['vergadering_datum'].min().date()} → {df['vergadering_datum'].max().date()}"
print("="*55)
print(f"  Total speeches analysed  : {total:,}")
print(f"  Motions passed           : {n_pass:,}  ({n_pass/total*100:.1f}%)")
print(f"  Motions rejected         : {n_rej:,}  ({n_rej/total*100:.1f}%)")
print(f"  Unique speakers          : {n_spk:,}")
print(f"  Unique parties           : {n_party}")
print(f"  Date range               : {dr}")
print("="*55)

## 2. Time-of-Day Patterns

In [ ]:
tod = df.groupby('time_bin')['label'].agg(['mean','count']).rename(columns={'mean':'pass_rate','count':'n'})
fig, axes = plt.subplots(2,2,figsize=(14,9))
fig.suptitle('Time-of-Day Analysis', fontsize=14, fontweight='bold', y=1.01)

colors_t=[GREEN if v>=0.5 else RED for v in tod['pass_rate']]
bars=axes[0,0].bar(tod.index, tod['pass_rate']*100, color=colors_t, edgecolor='white', alpha=0.85, width=0.6)
axes[0,0].axhline(50,color='grey',linestyle='--',lw=0.8); axes[0,0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0,0].set_title('Pass Rate by Time of Day',fontweight='bold'); axes[0,0].set_ylim(0,80); axes[0,0].tick_params(axis='x',rotation=25)
for bar,n in zip(bars,tod['n']): axes[0,0].text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.5,f'n={n}',ha='center',fontsize=7.5,color='#555')

tp=df[df['label']==1].groupby('time_bin').size(); tr_=df[df['label']==0].groupby('time_bin').size()
x2=range(len(tod.index))
axes[0,1].bar(x2, tp.reindex(tod.index,fill_value=0), label='Passed', color=GREEN, edgecolor='white', alpha=0.85)
axes[0,1].bar(x2, tr_.reindex(tod.index,fill_value=0), bottom=tp.reindex(tod.index,fill_value=0), label='Rejected', color=RED, edgecolor='white', alpha=0.85)
axes[0,1].set_xticks(list(x2)); axes[0,1].set_xticklabels(tod.index,rotation=25,ha='right')
axes[0,1].set_title('Volume by Time of Day',fontweight='bold'); axes[0,1].legend()

hr=df[df['hour']>=0].groupby('hour')['label'].mean()
axes[1,0].bar(hr.index, hr.values*100, color=BLUE, edgecolor='white', alpha=0.8)
axes[1,0].axhline(50,color=RED,linestyle='--',lw=0.8); axes[1,0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1,0].set_xlabel('Hour of Day'); axes[1,0].set_title('Pass Rate by Hour',fontweight='bold'); axes[1,0].set_xticks(range(0,24,2))

df_hm=df[(df['hour']>=0)&df['day_of_week'].notna()]
hm=df_hm.groupby(['hour','day_of_week'])['label'].mean().unstack(fill_value=np.nan)
hm.columns=['Mon','Tue','Wed','Thu','Fri','Sat','Sun'][:len(hm.columns)]
sns.heatmap(hm,ax=axes[1,1],cmap='RdYlGn',vmin=0,vmax=1,linewidths=0.3,linecolor='white',
            annot=True,fmt='.2f',cbar_kws={'label':'Pass Rate','format':'%.0%%'})
axes[1,1].set_title('Pass Rate: Hour × Day',fontweight='bold')
plt.tight_layout(); plt.show()

## 3. Per-Politician & Per-Party Analysis

In [ ]:
top_spk = df.groupby('speaker_name').agg(n=('speech_id','count'),pass_rate=('label','mean')).sort_values('n',ascending=False).head(20)
fig, axes = plt.subplots(1,2,figsize=(15,7))
axes[0].barh(top_spk.index[::-1], top_spk['n'][::-1], color=BLUE, edgecolor='white', alpha=0.85)
axes[0].set_title('Top 20 Speakers — Speech Count',fontweight='bold')
cols_s=[GREEN if v>=0.5 else RED for v in top_spk['pass_rate'][::-1]]
axes[1].barh(top_spk.index[::-1], top_spk['pass_rate'][::-1]*100, color=cols_s, edgecolor='white', alpha=0.85)
axes[1].axvline(50,color='grey',linestyle='--',lw=0.8); axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_title('Top 20 Speakers — Pass Rate',fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
party=df.groupby('speaker_party').agg(n=('speech_id','count'),pass_rate=('label','mean')).sort_values('n',ascending=False).head(15)
fig,axes=plt.subplots(1,2,figsize=(14,5))
axes[0].bar(party.index,party['n'],color=BLUE,edgecolor='white',alpha=0.85); axes[0].set_title('Speeches per Party',fontweight='bold'); axes[0].tick_params(axis='x',rotation=45)
cols_p=[GREEN if v>=0.5 else RED for v in party['pass_rate']]
axes[1].bar(party.index,party['pass_rate']*100,color=cols_p,edgecolor='white',alpha=0.85)
axes[1].axhline(50,color='grey',linestyle='--'); axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_title('Pass Rate per Party',fontweight='bold'); axes[1].tick_params(axis='x',rotation=45)
plt.tight_layout(); plt.show()

## 4. RobBERT Sentiment Analysis

In [ ]:
if not HAS_BERT:
    print("⚠️  Run Notebook 3 first.")
else:
    fig,axes=plt.subplots(1,3,figsize=(16,4)); fig.suptitle('RobBERT Sentiment',fontsize=13,fontweight='bold')
    counts=df['sentiment_label'].value_counts()
    axes[0].pie(counts,labels=counts.index,colors=[GREEN if l=='positive' else RED for l in counts.index],
                autopct='%1.1f%%',startangle=90,wedgeprops={'edgecolor':'white','linewidth':1.5})
    axes[0].set_title('Sentiment Distribution')
    axes[1].hist(df['sentiment_score'],bins=60,color=BLUE,edgecolor='white',alpha=0.85)
    axes[1].axvline(0,color=RED,linestyle='--'); axes[1].set_title('Score Distribution')
    acc_s=df.groupby('sentiment_label')['label'].mean()*100
    axes[2].bar(acc_s.index,acc_s.values,color=[GREEN if l=='positive' else RED for l in acc_s.index],edgecolor='white',alpha=0.85)
    axes[2].axhline(50,color='grey',linestyle='--'); axes[2].yaxis.set_major_formatter(mtick.PercentFormatter())
    axes[2].set_title('Pass Rate by Sentiment'); axes[2].set_ylim(0,75)
    plt.tight_layout(); plt.show()

In [ ]:
if HAS_BERT:
    top_p=df['speaker_party'].value_counts().head(12).index
    ps=df[df['speaker_party'].isin(top_p)].groupby('speaker_party')['sentiment_score'].mean().sort_values()
    fig,ax=plt.subplots(figsize=(8,6))
    ax.barh(ps.index,ps.values,color=[GREEN if v>=0 else RED for v in ps.values],edgecolor='white',alpha=0.85)
    ax.axvline(0,color='black',lw=0.7); ax.set_title('Mean RobBERT Sentiment Score by Party',fontweight='bold')
    plt.tight_layout(); plt.show()

In [ ]:
if HAS_BERT:
    df_ts=df.dropna(subset=['vergadering_datum','sentiment_score']).set_index('vergadering_datum')
    q_mean=df_ts['sentiment_score'].resample('QE').mean(); q_std=df_ts['sentiment_score'].resample('QE').std()
    fig,ax=plt.subplots(figsize=(13,3.5))
    ax.plot(q_mean.index,q_mean.values,color=BLUE,lw=2)
    ax.fill_between(q_mean.index,q_mean-q_std,q_mean+q_std,alpha=0.15,color=BLUE)
    ax.fill_between(q_mean.index,q_mean.values,0,where=(q_mean.values>0),alpha=0.2,color=GREEN)
    ax.fill_between(q_mean.index,q_mean.values,0,where=(q_mean.values<=0),alpha=0.2,color=RED)
    ax.axhline(0,color='grey',lw=0.8,linestyle='--'); ax.set_title('Sentiment Over Time (quarterly)',fontweight='bold')
    plt.tight_layout(); plt.show()

## 5. Tone — How It Changes Over the Day

In [ ]:
if HAS_BERT:
    tod_pivot=df.groupby(['time_bin','tone_label'])['speech_id'].count().unstack(fill_value=0)
    tod_pct=tod_pivot.div(tod_pivot.sum(axis=1),axis=0)*100
    fig,ax=plt.subplots(figsize=(11,5))
    tod_pct.plot(kind='bar',ax=ax,stacked=True,color=[TONE_COLORS.get(c,GREY) for c in tod_pct.columns],edgecolor='white',alpha=0.9)
    ax.set_title('How Tone Changes Over the Day — All Speakers (stacked %)',fontsize=13,fontweight='bold')
    ax.set_xlabel('Time of Day'); ax.set_ylabel('% of Speeches')
    ax.legend(title='Tone',bbox_to_anchor=(1.01,1),loc='upper left'); ax.tick_params(axis='x',rotation=25)
    plt.tight_layout(); plt.show()

In [ ]:
if HAS_BERT:
    top_spk_names=df['speaker_name'].value_counts().head(15).index
    df_agg=df[df['speaker_name'].isin(top_spk_names)].copy()
    agg_pivot=df_agg.groupby(['speaker_name','time_bin'])['tone_aggressive'].mean().unstack(fill_value=np.nan)
    fig,ax=plt.subplots(figsize=(11,6))
    sns.heatmap(agg_pivot,ax=ax,cmap='Reds',linewidths=0.4,linecolor='white',
                annot=True,fmt='.2f',cbar_kws={'label':'Mean Aggressive Score'})
    ax.set_title('Aggressive Tone Score — Top 15 Speakers × Time of Day',fontsize=12,fontweight='bold')
    plt.tight_layout(); plt.show()

In [ ]:
if HAS_BERT:
    tone_score_cols=['tone_aggressive','tone_mean','tone_neutral','tone_peaceful','tone_kind','tone_happy']
    tone_short=['Aggr.','Mean','Neutral','Peace.','Kind','Happy']
    top6=df['speaker_name'].value_counts().head(6).index
    fig,axes=plt.subplots(2,3,figsize=(15,8))
    fig.suptitle('Tone Profile per Speaker (mean score per tone)',fontsize=13,fontweight='bold')
    for ax,spk in zip(axes.flatten(),top6):
        subset=df[df['speaker_name']==spk]
        means=[subset[c].mean() for c in tone_score_cols if c in subset.columns]
        ax.bar(tone_short[:len(means)],means,color=[TONE_COLORS[t] for t in ['aggressive','mean','neutral','peaceful','kind','happy'][:len(means)]],edgecolor='white',alpha=0.9)
        party=subset['speaker_party'].mode()[0] if len(subset)>0 else ''
        ax.set_title(f"{spk}\n({party}, n={len(subset)})",fontsize=9,fontweight='bold')
        ax.set_ylim(0,1); ax.tick_params(axis='x',rotation=20)
    plt.tight_layout(); plt.show()

## 6. Model Comparison

In [ ]:
results_dir="outputs"; model_results={}
if os.path.isdir(results_dir):
    for fname in os.listdir(results_dir):
        if fname.endswith('_results.json'):
            with open(os.path.join(results_dir,fname)) as f: r=json.load(f); model_results[r['model']]=r

if model_results:
    comp=pd.DataFrame([{'Model':r['model'],'Accuracy':r.get('accuracy',np.nan),'F1':r.get('f1',np.nan),
                         'ROC-AUC':r.get('roc_auc',np.nan),'Brier↓':r.get('brier',np.nan)}
                        for r in model_results.values()]).sort_values('ROC-AUC',ascending=False)
    print(comp.to_string(index=False,float_format=lambda x: f'{x:.4f}'))
    metrics=['Accuracy','F1','ROC-AUC']; x=np.arange(3); w=0.22; colors_m=[BLUE,ORANGE,GREEN]
    fig,ax=plt.subplots(figsize=(10,5))
    for i,(_,row) in enumerate(comp.iterrows()):
        vals=[row['Accuracy'],row['F1'],row['ROC-AUC']]
        bars=ax.bar(x+i*w,vals,w,label=row['Model'],color=colors_m[i%3],alpha=0.85,edgecolor='white')
        for bar in bars: ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.004,f"{bar.get_height():.3f}",ha='center',fontsize=8)
    ax.set_xticks(x+w); ax.set_xticklabels(metrics); ax.set_ylim(0,1.1); ax.legend()
    ax.set_title('Model Comparison — Test Set',fontsize=13,fontweight='bold'); ax.axhline(0.5,color='grey',linestyle=':',lw=0.8)
    plt.tight_layout(); plt.show()
else:
    print("⚠️  No results found — run Notebook 4 first.")

## 7. Launch Interactive Streamlit Dashboard

```bash
pip install streamlit plotly
streamlit run dashboard_app.py -- --data speeches_final.csv
```

The dashboard includes all the above charts with interactive filters:
- Filter by speaker, party, date range, motion outcome
- Per-speaker tone-over-time line charts
- Live model metrics loaded from `outputs/*.json`


In [ ]:
print("\n🏛  DUTCH PARLIAMENT SPEECHES PIPELINE — SUMMARY REPORT")
print("="*60)
print(f"  Total speeches        : {total:,}")
print(f"  Pass rate             : {n_pass/total*100:.1f}%")
print(f"  Unique speakers       : {n_spk}")
print(f"  Unique parties        : {n_party}")
print(f"  Date range            : {dr}")
if HAS_BERT:
    print(f"  Dominant sentiment    : {df['sentiment_label'].value_counts().idxmax()}")
    print(f"  Dominant tone         : {df['tone_label'].value_counts().idxmax()}")
    print(f"  Most positive party   : {df.groupby('speaker_party')['sentiment_score'].mean().idxmax()}")
if model_results:
    best=max(model_results.values(),key=lambda r:r.get('roc_auc',0))
    print(f"  Best model (AUC)      : {best['model']} ({best['roc_auc']:.4f})")
print("="*60)